# Setup — Catalog e Schemas (Unity Catalog)

Cria, de forma idempotente (`CREATE ... IF NOT EXISTS`), a estrutura de governança de
dados do projeto: 1 catalog e 5 schemas (um por camada da arquitetura Medallion, mais
o schema de reconciliação).

Este notebook documenta como código a infraestrutura que, até o schema `gold`, havia
sido criada manualmente pela interface do Catalog Explorer — reexecutá-lo não recria
nada que já existe, apenas garante que a estrutura completa (incluindo `reconciliation`,
criado a partir daqui) esteja presente.

**Catalog:** `poc_b3_modernizacao`
**Schemas:** `landing`, `bronze`, `silver`, `gold`, `reconciliation`

In [0]:
# imports
from pyspark.sql import functions as F

In [0]:
# cria catalog (idempotente)
spark.sql("""
    CREATE CATALOG IF NOT EXISTS poc_b3_modernizacao
    COMMENT 'Catalog do projeto de portfolio de modernizacao de dados B3 (KNIME -> Databricks). Simula migracao de pipeline legado para arquitetura Medallion, com calculo de indice-proxy sobre 4 tickers via API brapi.dev.'
""")

spark.sql("""
    ALTER CATALOG poc_b3_modernizacao
    SET TAGS ('ambiente' = 'poc', 'dominio' = 'mercado-financeiro', 'projeto' = 'b3-modernizacao-dados')
""")

print("Catalog poc_b3_modernizacao pronto.")

In [0]:
# cria os 5 schemas (idempotente)
schemas = {
    "landing": "Camada de pouso (landing zone) - copia bruta e imutavel da resposta da API brapi.dev, antes de qualquer parsing ou tipagem. Ver ADR-01.",
    "bronze": "Copia fiel da fonte (Volume UC landing), imutavel, com metadados de ingestao. Sem tratamento ou tipagem de conteudo.",
    "silver": "Dados tipados, deduplicados, com regra de qualidade e tabela de quarentena para registros invalidos.",
    "gold": "Calculo do indice-proxy e indicadores, gerado de forma automatizada via Job para eliminar divergencia de numeros entre consumidores.",
    "reconciliation": "Resultado da comparacao entre a Gold (Databricks) e a saida do KNIME (sistema legado simulado), por data - prova de migracao de logica de negocio preservada.",
}

for nome_schema, comentario in schemas.items():
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS poc_b3_modernizacao.{nome_schema}
        COMMENT '{comentario}'
    """)
    spark.sql(f"""
        ALTER SCHEMA poc_b3_modernizacao.{nome_schema}
        SET TAGS ('ambiente' = 'poc', 'dominio' = 'mercado-financeiro', 'projeto' = 'b3-modernizacao-dados', 'camada' = '{nome_schema}')
    """)
    print(f"Schema {nome_schema} pronto.")